# Speech Emotion Recognition on RAVDESS: GPU training

*A model that recognises 8 emotions from voice, tested only on actors it never heard.*

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Every run writes its own JSON to
`results/runs/`, and finished runs are skipped, so if Colab disconnects, reconnect and run all again.

Generated by `tools/build_colab_notebook.py`. Edit the files in the repo, not these cells.

In [ ]:
import os, subprocess
os.makedirs('/content/ser_project/ser', exist_ok=True)
os.makedirs('/content/ser_project/scripts', exist_ok=True)
os.chdir('/content/ser_project')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
import torch, transformers, librosa
print('torch', torch.__version__, '| transformers', transformers.__version__, '| librosa', librosa.__version__,
      '| cuda', torch.cuda.is_available())

def run(args):
    # Run one experiment stage, stream its log, and stop the notebook if it fails.
    p = subprocess.Popen(f'python -u scripts/run_experiments.py {args}', shell=True, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'stage failed: {args}')

In [ ]:
%%writefile ser/__init__.py

In [ ]:
%%writefile ser/data.py
"""Dataset download, metadata parsing, audio loading and speaker-independent splits."""
from __future__ import annotations

import json
import pickle
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SR = 16000
SPLIT_SEED = 1234  # fixed once, never tuned

# ---------------------------------------------------------------- RAVDESS
RAVDESS_URL = "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
# Filename field 3 is the emotion code 01..08, in this order.
RAVDESS_EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
ACTOR_SPLIT = {"train": range(1, 19), "val": range(19, 21), "test": range(21, 25)}

# ---------------------------------------------------------------- CREMA-D
CREMAD_REPO = "CheyneyComputerScience/CREMA-D"
CREMAD_MEDIA = f"https://media.githubusercontent.com/media/{CREMAD_REPO}/master/AudioWAV/"
# Fallback: byte-identical AudioWAV files packed as one archive on the Hugging Face Hub.
CREMAD_MIRROR = "https://huggingface.co/datasets/myleslinder/crema-d/resolve/main/data/crema_d.tar.gz"
CREMAD_CODES = {"NEU": "neutral", "HAP": "happy", "SAD": "sad", "ANG": "angry", "FEA": "fearful", "DIS": "disgust"}
# The six emotions CREMA-D and RAVDESS have in common (RAVDESS adds calm and surprised).
SHARED_EMOTIONS = ["neutral", "happy", "sad", "angry", "fearful", "disgust"]
CREMAD_SPEAKER_SPLIT = (64, 9, 18)  # train / val / test speakers out of 91


def _download(url: str, dest: Path, retries: int = 6) -> None:
    for attempt in range(retries):
        try:
            tmp = dest.with_suffix(dest.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            tmp.replace(dest)
            return
        except Exception:  # network hiccup: back off and retry
            if attempt == retries - 1:
                raise
            time.sleep(min(2 ** attempt, 20))


def download_ravdess(root: str | Path) -> Path:
    root = Path(root)
    out = root / "ravdess"
    if len(list(out.glob("Actor_*/*.wav"))) == 1440:
        return out
    root.mkdir(parents=True, exist_ok=True)
    zpath = root / "ravdess_speech.zip"
    if not zpath.exists():
        print("Downloading RAVDESS speech audio from Zenodo (~200 MB)...")
        _download(RAVDESS_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(out)
    return out


def ravdess_metadata(root: str | Path) -> pd.DataFrame:
    rows = []
    for p in sorted(Path(root).glob("Actor_*/*.wav")):
        f = p.stem.split("-")  # modality-channel-emotion-intensity-statement-repetition-actor
        actor = int(f[6])
        rows.append(dict(path=str(p), dataset="ravdess", speaker=actor,
                         emotion=RAVDESS_EMOTIONS[int(f[2]) - 1], intensity=int(f[3]),
                         statement=int(f[4]), gender="male" if actor % 2 else "female"))
    df = pd.DataFrame(rows)
    assert len(df) == 1440, f"expected 1440 RAVDESS clips, found {len(df)}"
    return df


def ravdess_split(df: pd.DataFrame, kind: str) -> dict[str, np.ndarray]:
    """Return index arrays for train/val/test.

    actor : actors 1-18 train, 19-20 val, 21-24 test (no speaker overlap)
    random: stratified random split with exactly the same sizes (1080/120/240)
    """
    if kind == "actor":
        return {k: df.index[df.speaker.isin(list(v))].to_numpy() for k, v in ACTOR_SPLIT.items()}
    if kind == "random":
        idx = df.index.to_numpy()
        trval, test = train_test_split(idx, test_size=240, stratify=df.emotion, random_state=SPLIT_SEED)
        train, val = train_test_split(trval, test_size=120, stratify=df.emotion[trval], random_state=SPLIT_SEED)
        return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}
    raise ValueError(kind)


def download_cremad(root: str | Path, workers: int = 8, limit: int | None = None) -> Path:
    """Fetch the 7,442 CREMA-D WAV files from the official GitHub LFS store.

    `limit` keeps an evenly spaced subset (smoke tests only); the metadata
    reader then only sees the files that were downloaded.
    """
    out = Path(root) / "cremad"
    out.mkdir(parents=True, exist_ok=True)
    api = f"https://api.github.com/repos/{CREMAD_REPO}"
    listing = out / "_listing.json"
    if not listing.exists():
        with urllib.request.urlopen(f"{api}/contents/") as r:
            sha = next(d["sha"] for d in json.load(r) if d["name"] == "AudioWAV")
        with urllib.request.urlopen(f"{api}/git/trees/{sha}") as r:
            names = [t["path"] for t in json.load(r)["tree"] if t["path"].endswith(".wav")]
        listing.write_text(json.dumps(names))
    names = json.loads(listing.read_text())
    if limit:
        names = names[:: max(1, len(names) // limit)]
    todo = [n for n in names if not (out / n).exists() or (out / n).stat().st_size < 1000]
    if todo:
        print(f"Downloading {len(todo)} CREMA-D files from GitHub LFS...", flush=True)
        try:
            with ThreadPoolExecutor(workers) as ex:
                list(ex.map(lambda n: _download(CREMAD_MEDIA + n, out / n), todo))
        except Exception as e:  # GitHub LFS sometimes answers 503 under load
            print(f"GitHub LFS failed ({e}); using the Hugging Face mirror of the same files.", flush=True)
            _cremad_from_mirror(out)
    return out


def _cremad_from_mirror(out: Path) -> None:
    import tarfile
    tgz = out.parent / "crema_d.tar.gz"
    if not tgz.exists():
        _download(CREMAD_MIRROR, tgz)
    with tarfile.open(tgz) as tar:
        for m in tar:
            if m.isfile() and m.name.endswith(".wav"):
                m.name = Path(m.name).name
                tar.extract(m, out, filter="data")


def cremad_metadata(root: str | Path) -> pd.DataFrame:
    rows = []
    listing = Path(root) / "_listing.json"
    official = set(json.loads(listing.read_text())) if listing.exists() else None
    for p in sorted(Path(root).glob("*.wav")):
        if official is not None and p.name not in official:  # e.g. a misnamed duplicate in the mirror
            continue
        spk, sent, emo, level = p.stem.split("_")
        if p.stat().st_size < 1000:  # a handful of CREMA-D files are empty
            continue
        rows.append(dict(path=str(p), dataset="cremad", speaker=int(spk), emotion=CREMAD_CODES[emo],
                         intensity=level, statement=sent))
    return pd.DataFrame(rows)


def cremad_split(df: pd.DataFrame) -> dict[str, np.ndarray]:
    speakers = np.array(sorted(df.speaker.unique()))
    perm = np.random.default_rng(SPLIT_SEED).permutation(speakers)
    n_tr, n_va, _ = CREMAD_SPEAKER_SPLIT
    groups = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    return {k: df.index[df.speaker.isin(v)].to_numpy() for k, v in groups.items()}


# ---------------------------------------------------------------- audio
def preprocess_wave(y: np.ndarray, sr: int, top_db: float = 40.0) -> np.ndarray:
    """Mono 16 kHz, leading/trailing silence trimmed, peak-normalised.

    The same function is used for training data and for the live demo, so a
    microphone recording goes through exactly the pipeline the model saw.
    """
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=-1) if y.shape[-1] <= 8 else y.mean(axis=0)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    if len(trimmed) >= 0.3 * SR:
        y = trimmed
    peak = np.abs(y).max()
    if peak > 0:
        y = 0.95 * y / peak
    return y.astype(np.float32)


def load_wave(path: str) -> np.ndarray:
    y, sr = librosa.load(path, sr=None, mono=True)
    return preprocess_wave(y, sr)


def load_waves(paths: list[str], cache: str | Path | None = None, workers: int = 4) -> list[np.ndarray]:
    """Load (and cache as int16) every clip; returns float32 arrays."""
    if cache is not None and Path(cache).exists():
        with open(cache, "rb") as f:
            stored = pickle.load(f)
        if stored["paths"] == list(paths):
            return [w.astype(np.float32) / 32767.0 for w in stored["waves"]]
    with ThreadPoolExecutor(workers) as ex:
        waves = list(ex.map(load_wave, paths))
    if cache is not None:
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        with open(cache, "wb") as f:
            pickle.dump({"paths": list(paths), "waves": [(w * 32767).astype(np.int16) for w in waves]}, f)
    return waves

In [ ]:
%%writefile ser/models.py
"""MFCC-CNN baseline and a self-supervised (WavLM / wav2vec2) classifier."""
from __future__ import annotations

import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import SR

# ---------------------------------------------------------------- MFCC baseline
N_MFCC = 40
HOP = 160  # 10 ms


def mfcc_features(y: np.ndarray) -> np.ndarray:
    """(3, 40, T): MFCCs plus first and second deltas, 25 ms window, 10 ms hop."""
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=512, win_length=400, hop_length=HOP, n_mels=64)
    if m.shape[1] < 9:  # delta needs at least 9 frames
        m = np.pad(m, ((0, 0), (0, 9 - m.shape[1])), mode="edge")
    return np.stack([m, librosa.feature.delta(m), librosa.feature.delta(m, order=2)]).astype(np.float32)


class MFCCCNN(nn.Module):
    def __init__(self, n_classes: int, width: int = 32, dropout: float = 0.3):
        super().__init__()
        chans, layers, c = [width, width * 2, width * 4, width * 4], [], 3
        for o in chans:
            layers += [nn.Conv2d(c, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                       nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2), nn.Dropout2d(0.1)]
            c = o
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c, n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # x: (B, 3, 40, T)
        h = self.features(x).mean(2)  # average over frequency -> (B, C, T')
        return self.head(torch.cat([h.mean(-1), h.amax(-1)], dim=1))


# ---------------------------------------------------------------- SSL model
SSL_MODEL = "microsoft/wavlm-base-plus"


def conv_output_lengths(config, lengths: torch.Tensor) -> torch.Tensor:
    for k, s in zip(config.conv_kernel, config.conv_stride):
        lengths = torch.div(lengths - k, s, rounding_mode="floor") + 1
    return lengths


def masked_layer_means(hidden_states, frame_mask: torch.Tensor | None) -> torch.Tensor:
    """Tuple of L (B, T, H) tensors -> (B, L, H) time-averaged over valid frames."""
    h = torch.stack(hidden_states, dim=1).float()  # (B, L, T, H)
    if frame_mask is None:
        return h.mean(2)
    m = frame_mask[:, None, :, None].to(h.dtype)
    return (h * m).sum(2) / m.sum(2).clamp(min=1)


class LayerWeightedHead(nn.Module):
    """SUPERB-style learnable softmax weighting of all layers + small MLP.

    Operates on per-layer, time-averaged vectors. Because averaging is linear,
    this equals weighting frame-level layers and then averaging, which lets the
    frozen model be trained on cached features in seconds.
    """

    def __init__(self, n_layers: int, hidden: int, n_classes: int, proj: int = 256, dropout: float = 0.2):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.mlp = nn.Sequential(nn.Linear(hidden, proj), nn.ReLU(), nn.Dropout(dropout), nn.Linear(proj, n_classes))

    def forward(self, pooled: torch.Tensor) -> torch.Tensor:  # (B, L, H)
        assert pooled.shape[1] == len(self.layer_logits), (
            f"got {pooled.shape[1]} layers, expected {len(self.layer_logits)} (is LayerDrop on?)")
        pooled = F.layer_norm(pooled, pooled.shape[-1:])
        w = torch.softmax(self.layer_logits, 0)
        return self.mlp((w[None, :, None] * pooled).sum(1))

    def layer_weights(self) -> list[float]:
        return torch.softmax(self.layer_logits.detach().cpu(), 0).tolist()


class SSLClassifier(nn.Module):
    def __init__(self, n_classes: int, model_name: str = SSL_MODEL, encoder: nn.Module | None = None):
        super().__init__()
        if encoder is None:
            from transformers import AutoModel
            encoder = AutoModel.from_pretrained(model_name)
        # LayerDrop would randomly skip transformer layers while training, so
        # output_hidden_states would return fewer than num_hidden_layers + 1 entries
        # and the fixed layer-weight vector would no longer line up.
        encoder.config.layerdrop = 0.0
        self.encoder = encoder
        cfg = encoder.config
        self.head = LayerWeightedHead(cfg.num_hidden_layers + 1, cfg.hidden_size, n_classes)

    def freeze_feature_encoder(self) -> None:
        for p in self.encoder.feature_extractor.parameters():
            p.requires_grad = False

    def pooled(self, wav: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        out = self.encoder(wav, attention_mask=attention_mask, output_hidden_states=True)
        frame_mask = None
        if attention_mask is not None:
            n = conv_output_lengths(self.encoder.config, attention_mask.sum(-1))
            T = out.hidden_states[0].shape[1]
            frame_mask = torch.arange(T, device=wav.device)[None, :] < n[:, None]
        return masked_layer_means(out.hidden_states, frame_mask)

    def forward(self, wav: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        return self.head(self.pooled(wav, attention_mask))

In [ ]:
%%writefile ser/train.py
"""Training / evaluation loops. Model selection uses validation UA only."""
from __future__ import annotations

import copy
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score

from .data import SR
from .models import MFCCCNN, SSLClassifier


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def metrics(y_true, y_pred, n_classes: int) -> dict:
    """UA = unweighted accuracy = mean per-class recall; WA = plain accuracy."""
    return dict(ua=float(balanced_accuracy_score(y_true, y_pred)),
                wa=float(accuracy_score(y_true, y_pred)),
                macro_f1=float(f1_score(y_true, y_pred, average="macro")),
                confusion=confusion_matrix(y_true, y_pred, labels=list(range(n_classes))).tolist())


def class_weights(y: np.ndarray, n_classes: int, device) -> torch.Tensor:
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    w = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(w, device=device)


def restricted_argmax(logits: np.ndarray, keep: list[int]) -> np.ndarray:
    """Predict only among `keep` columns; returns positions within `keep`."""
    return logits[:, keep].argmax(1)


# ================================================================ MFCC-CNN
CNN_FRAMES = 350  # 3.5 s


def _crop_pad(x: np.ndarray, T: int, rng: np.random.Generator | None) -> np.ndarray:
    t = x.shape[-1]
    if t >= T:
        s = int(rng.integers(0, t - T + 1)) if rng is not None else (t - T) // 2
        return x[..., s:s + T]
    out = np.zeros((*x.shape[:-1], T), np.float32)
    s = int(rng.integers(0, T - t + 1)) if rng is not None else 0
    out[..., s:s + t] = x
    return out


def _spec_augment(x: torch.Tensor, rng: np.random.Generator) -> torch.Tensor:
    B, _, F_, T = x.shape
    x = x.clone()
    for b in range(B):
        f = int(rng.integers(0, 7))
        f0 = int(rng.integers(0, F_ - f + 1))
        x[b, :, f0:f0 + f, :] = 0
        for _ in range(2):
            t = int(rng.integers(0, 41))
            t0 = int(rng.integers(0, T - t + 1))
            x[b, :, :, t0:t0 + t] = 0
    return x


class CNNTrainer:
    def __init__(self, n_classes: int, device, epochs: int = 60, patience: int = 15, batch_size: int = 32,
                 lr: float = 1e-3):
        self.n_classes, self.device, self.epochs, self.patience = n_classes, device, epochs, patience
        self.batch_size, self.lr = batch_size, lr

    def _norm(self, feats):
        return [(f - self.mean) / self.std for f in feats]

    def _batch(self, feats, idx, rng):
        return torch.from_numpy(np.stack([_crop_pad(feats[i], CNN_FRAMES, rng) for i in idx])).to(self.device)

    def logits(self, feats) -> np.ndarray:
        self.model.eval()
        feats, out = self._norm(feats), []
        with torch.no_grad():
            for s in range(0, len(feats), 64):
                out.append(self.model(self._batch(feats, range(s, min(s + 64, len(feats))), None)).cpu().numpy())
        return np.concatenate(out)

    def fit(self, tr_feats, tr_y, va_feats, va_y, seed: int) -> dict:
        set_seed(seed)
        rng = np.random.default_rng(seed)
        cat = np.concatenate(tr_feats, axis=-1)  # normalisation stats from training data only
        self.mean, self.std = cat.mean(-1, keepdims=True), cat.std(-1, keepdims=True) + 1e-5
        tr = self._norm(tr_feats)
        self.model = MFCCCNN(self.n_classes).to(self.device)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.epochs)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights(tr_y, self.n_classes, self.device))
        y_t = torch.from_numpy(tr_y).to(self.device)
        best, best_state, best_epoch, bad, hist = -1.0, None, 0, 0, []
        for ep in range(self.epochs):
            self.model.train()
            perm = rng.permutation(len(tr))
            for s in range(0, len(perm), self.batch_size):
                idx = perm[s:s + self.batch_size]
                x = _spec_augment(self._batch(tr, idx, rng), rng)
                loss = loss_fn(self.model(x), y_t[idx])
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()
            va_ua = metrics(va_y, self.logits(va_feats).argmax(1), self.n_classes)["ua"]
            hist.append(va_ua)
            if va_ua > best:
                best, best_epoch, bad = va_ua, ep, 0
                best_state = copy.deepcopy(self.model.state_dict())
            else:
                bad += 1
                if bad >= self.patience:
                    break
        self.model.load_state_dict(best_state)
        return dict(val_ua=best, best_epoch=best_epoch, epochs_run=len(hist), val_history=hist)


# ================================================================ SSL: frozen
@torch.no_grad()
def extract_pooled(encoder, waves, device, max_sec: float = 10.0, log_every: int = 500) -> np.ndarray:
    """Per-layer time-averaged hidden states, one clip at a time (no padding)."""
    model = SSLClassifier(2, encoder=encoder).to(device).eval()
    use_amp = torch.device(device).type == "cuda"
    out = []
    for i, w in enumerate(waves):
        x = torch.from_numpy(w[: int(max_sec * SR)])[None].to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out.append(model.pooled(x).squeeze(0).cpu().numpy().astype(np.float16))
        if log_every and (i + 1) % log_every == 0:
            print(f"  extracted {i + 1}/{len(waves)}", flush=True)
    return np.stack(out)


class HeadTrainer:
    def __init__(self, n_layers: int, hidden: int, n_classes: int, device, epochs: int = 150, patience: int = 25,
                 batch_size: int = 32, lr: float = 1e-3):
        from .models import LayerWeightedHead
        self.make = lambda: LayerWeightedHead(n_layers, hidden, n_classes)
        self.n_classes, self.device, self.epochs, self.patience = n_classes, device, epochs, patience
        self.batch_size, self.lr = batch_size, lr

    def logits(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        with torch.no_grad():
            return self.model(torch.from_numpy(X).float().to(self.device)).cpu().numpy()

    def fit(self, Xtr, ytr, Xva, yva, seed: int) -> dict:
        set_seed(seed)
        rng = np.random.default_rng(seed)
        self.model = self.make().to(self.device)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights(ytr, self.n_classes, self.device))
        X = torch.from_numpy(Xtr).float().to(self.device)
        y = torch.from_numpy(ytr).to(self.device)
        best, best_state, best_epoch, bad, hist = -1.0, None, 0, 0, []
        for ep in range(self.epochs):
            self.model.train()
            perm = rng.permutation(len(X))
            for s in range(0, len(perm), self.batch_size):
                idx = torch.from_numpy(perm[s:s + self.batch_size]).to(self.device)
                loss = loss_fn(self.model(X[idx]), y[idx])
                opt.zero_grad()
                loss.backward()
                opt.step()
            va_ua = metrics(yva, self.logits(Xva).argmax(1), self.n_classes)["ua"]
            hist.append(va_ua)
            if va_ua > best:
                best, best_epoch, bad = va_ua, ep, 0
                best_state = copy.deepcopy(self.model.state_dict())
            else:
                bad += 1
                if bad >= self.patience:
                    break
        self.model.load_state_dict(best_state)
        return dict(val_ua=best, best_epoch=best_epoch, epochs_run=len(hist), val_history=hist,
                    layer_weights=self.model.layer_weights())


# ================================================================ SSL: fine-tuning
def _length_bucketed_batches(lengths: np.ndarray, batch_size: int, rng: np.random.Generator) -> list:
    perm = rng.permutation(len(lengths))
    batches = []
    for s in range(0, len(perm), batch_size * 50):
        chunk = perm[s:s + batch_size * 50]
        chunk = chunk[np.argsort(lengths[chunk])]
        batches += [chunk[i:i + batch_size] for i in range(0, len(chunk), batch_size)]
    rng.shuffle(batches)
    return batches


class FinetuneTrainer:
    def __init__(self, n_classes: int, device, model_name: str, epochs: int = 20, batch_size: int = 8,
                 lr_encoder: float = 5e-5, lr_head: float = 1e-3, max_train_sec: float = 5.0, eval_every: int = 1,
                 freeze_epochs: int = 0):
        self.n_classes, self.device, self.model_name, self.epochs = n_classes, device, model_name, epochs
        self.batch_size, self.lr_encoder, self.lr_head = batch_size, lr_encoder, lr_head
        self.freeze_epochs = freeze_epochs  # train the head alone for this many epochs first
        self.max_train = int(max_train_sec * SR)
        self.eval_every = eval_every
        self.amp = torch.device(device).type == "cuda"

    @torch.no_grad()
    def logits(self, waves, max_sec: float = 10.0) -> np.ndarray:
        """Batch size 1: no padding, identical to how the demo scores a recording."""
        self.model.eval()
        out = []
        for w in waves:
            x = torch.from_numpy(w[: int(max_sec * SR)])[None].to(self.device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
                out.append(self.model(x).float().cpu().numpy())
        return np.concatenate(out)

    def _collate(self, waves, idx, rng):
        clips = []
        for i in idx:
            w = waves[i]
            if len(w) > self.max_train:
                s = int(rng.integers(0, len(w) - self.max_train + 1))
                w = w[s:s + self.max_train]
            clips.append(w)
        T = max(len(c) for c in clips)
        x = np.zeros((len(clips), T), np.float32)
        m = np.zeros((len(clips), T), np.int64)
        for j, c in enumerate(clips):
            x[j, :len(c)], m[j, :len(c)] = c, 1
        return torch.from_numpy(x).to(self.device), torch.from_numpy(m).to(self.device)

    def fit(self, tr_waves, tr_y, va_waves, va_y, seed: int, log=print) -> dict:
        set_seed(seed)
        rng = np.random.default_rng(seed)
        self.model = SSLClassifier(self.n_classes, self.model_name).to(self.device)
        self.model.freeze_feature_encoder()
        enc = [p for n, p in self.model.named_parameters() if n.startswith("encoder.") and p.requires_grad]
        opt = torch.optim.AdamW([{"params": enc, "lr": self.lr_encoder},
                                 {"params": self.model.head.parameters(), "lr": self.lr_head}], weight_decay=0.01)
        lengths = np.array([min(len(w), self.max_train) for w in tr_waves])
        steps_per_epoch = math.ceil(len(tr_waves) / self.batch_size)
        total, warmup = steps_per_epoch * self.epochs, max(1, int(0.1 * steps_per_epoch * self.epochs))
        freeze_steps = self.freeze_epochs * steps_per_epoch

        def shape(s):  # linear warm-up then linear decay
            return (s + 1) / warmup if s < warmup else max(0.0, (total - s) / max(1, total - warmup))

        sched = torch.optim.lr_scheduler.LambdaLR(opt, [lambda s: 0.0 if s < freeze_steps else shape(s), shape])
        scaler = torch.amp.GradScaler("cuda", enabled=self.amp)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights(tr_y, self.n_classes, self.device))
        y_t = torch.from_numpy(tr_y).to(self.device)
        best, best_state, best_epoch, hist = -1.0, None, 0, []
        for ep in range(self.epochs):
            self.model.train()
            for p in enc:  # head-only warm-up keeps the pretrained encoder intact at first
                p.requires_grad = ep >= self.freeze_epochs
            t0, run_loss = time.time(), 0.0
            batches = _length_bucketed_batches(lengths, self.batch_size, rng)
            for idx in batches:
                x, m = self._collate(tr_waves, idx, rng)
                with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
                    logits = self.model(x, m)
                loss = loss_fn(logits.float(), y_t[idx])
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                run_loss += loss.item()
            if (ep + 1) % self.eval_every and ep != self.epochs - 1:
                continue
            va_ua = metrics(va_y, self.logits(va_waves).argmax(1), self.n_classes)["ua"]
            hist.append(va_ua)
            log(f"  epoch {ep + 1}/{self.epochs} loss {run_loss / len(batches):.3f} val UA {va_ua:.3f} "
                f"({time.time() - t0:.0f}s)")
            if va_ua > best:
                best, best_epoch = va_ua, ep
                best_state = {k: v.detach().to("cpu", copy=True) for k, v in self.model.state_dict().items()}
        self.model.load_state_dict(best_state)
        return dict(val_ua=best, best_epoch=best_epoch, epochs_run=self.epochs, val_history=hist,
                    layer_weights=self.model.head.layer_weights())

In [ ]:
%%writefile scripts/run_experiments.py
"""Run every experiment in the project and write one JSON file per run.

Stages
  prepare   download RAVDESS (+ CREMA-D) and cache 16 kHz trimmed audio
  cnn       MFCC-CNN baseline            RAVDESS, actor + random split
  frozen    frozen WavLM + layer-weighted head      "
  finetune  fine-tuned WavLM                        "
  cremad    cross-corpus: WavLM trained on CREMA-D (speaker-independent),
            tested on unseen CREMA-D speakers and on all of RAVDESS

Runs that already have a result file are skipped, so the script can be
re-launched after a Colab disconnect. Test sets are only scored once, with the
checkpoint chosen on validation UA.
"""
from __future__ import annotations

import argparse
import json
import pickle
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import numpy as np
import torch

from ser import data as D
from ser.models import SSL_MODEL, mfcc_features
from ser.train import (CNNTrainer, FinetuneTrainer, HeadTrainer, extract_pooled, metrics, restricted_argmax)

RAV_SHARED = [D.RAVDESS_EMOTIONS.index(e) for e in D.SHARED_EMOTIONS]


class Workspace:
    def __init__(self, args):
        self.args = args
        self.root = Path(args.data_root)
        self.cache = self.root / "cache"
        self.out = Path(args.out)
        (self.out / "runs").mkdir(parents=True, exist_ok=True)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.slug = args.model_name.split("/")[-1]
        self._pooled = {}
        self._encoder = None
        self.quick_take = (lambda idx, k: np.sort(np.random.default_rng(0).permutation(idx)[:k])) if args.quick else None

    # ------------------------------------------------------------ data
    def ravdess(self):
        if not hasattr(self, "rav"):
            self.rav = D.ravdess_metadata(D.download_ravdess(self.root))
            self.rav_y = self.rav.emotion.map(D.RAVDESS_EMOTIONS.index).to_numpy()
            self.rav_waves = D.load_waves(self.rav.path.tolist(), self.cache / "ravdess_waves.pkl")
            shared = self.rav.emotion.isin(D.SHARED_EMOTIONS).to_numpy()
            self.rav6_idx = np.flatnonzero(shared)
            self.rav6_y = self.rav.emotion[shared].map(D.SHARED_EMOTIONS.index).to_numpy()
        return self.rav

    def cremad(self):
        if not hasattr(self, "crem"):
            root = D.download_cremad(self.root, limit=300 if self.args.quick else None)
            self.crem = D.cremad_metadata(root)
            self.crem_y = self.crem.emotion.map(D.SHARED_EMOTIONS.index).to_numpy()
            name = "cremad_waves_quick.pkl" if self.args.quick else "cremad_waves.pkl"
            self.crem_waves = D.load_waves(self.crem.path.tolist(), self.cache / name)
            self.crem_split = D.cremad_split(self.crem)
        return self.crem

    def split(self, kind):
        s = D.ravdess_split(self.ravdess(), kind) if kind != "cremad" else self.crem_split
        if self.quick_take:
            s = {k: self.quick_take(v, {"train": 96, "val": 24, "test": 32}[k]) for k, v in s.items()}
        return s

    def use_cremad(self):
        return not self.args.no_cremad

    # ------------------------------------------------------------ SSL features
    def encoder(self):
        if self._encoder is None:
            from transformers import AutoModel
            self._encoder = AutoModel.from_pretrained(self.args.model_name)
        return self._encoder

    def pooled(self, dataset):
        """(N, L, H) cached frozen features for every clip of a dataset."""
        if dataset not in self._pooled:
            suffix = "_quick" if self.args.quick else ""
            f = self.cache / f"{dataset}_pooled_{self.slug}{suffix}.npy"
            waves = self.rav_waves if dataset == "ravdess" else self.crem_waves
            if f.exists() and len(np.load(f, mmap_mode="r")) == len(waves):
                self._pooled[dataset] = np.load(f)
            else:
                print(f"Extracting frozen {self.slug} features for {len(waves)} {dataset} clips...", flush=True)
                if self.args.quick:  # only the clips a quick run touches
                    need = set()
                    for kind in (["actor", "random"] if dataset == "ravdess" else ["cremad"]):
                        for v in self.split(kind).values():
                            need.update(v.tolist())
                    if dataset == "ravdess":
                        need.update(self.rav6_idx[:32].tolist())
                    X = np.zeros((len(waves), self.encoder().config.num_hidden_layers + 1,
                                  self.encoder().config.hidden_size), np.float16)
                    need = sorted(need)
                    X[need] = extract_pooled(self.encoder(), [waves[i] for i in need], self.device)
                else:
                    X = extract_pooled(self.encoder(), waves, self.device)
                f.parent.mkdir(parents=True, exist_ok=True)
                np.save(f, X)
                self._pooled[dataset] = X
        return self._pooled[dataset]

    # ------------------------------------------------------------ bookkeeping
    def path(self, name):
        return self.out / "runs" / f"{name}.json"

    def done(self, name):
        return self.path(name).exists()

    def save(self, name, rec):
        rec = dict(name=name, **rec, quick=self.args.quick, device=self.device,
                   finished=time.strftime("%Y-%m-%d %H:%M:%S"))
        self.path(name).write_text(json.dumps(rec, indent=1))
        c = " ".join(f"{k}={v['ua']:.3f}" for k, v in rec.get("cross", {}).items())
        print(f"[done] {name}: val UA {rec['val_ua']:.3f} | test UA {rec['test']['ua']:.3f} "
              f"WA {rec['test']['wa']:.3f} {c}", flush=True)
        # The full record also goes to the log, so a notebook's saved output keeps every
        # result even if the machine that wrote results/runs/ is recycled.
        print("RESULT_JSON " + json.dumps(rec, separators=(",", ":")), flush=True)


def cross_eval_ravdess_model(ws, logits_fn):
    """8-class RAVDESS model scored on unseen CREMA-D speakers (6 shared emotions)."""
    if not ws.use_cremad():
        return {}
    ws.cremad()
    te = ws.split("cremad")["test"]
    pred = restricted_argmax(logits_fn(te), RAV_SHARED)
    return {"cremad_test": metrics(ws.crem_y[te], pred, 6)}


# ================================================================ stages
def stage_prepare(ws):
    df = ws.ravdess()
    print(f"RAVDESS: {len(df)} clips, {df.speaker.nunique()} actors")
    for kind in ["actor", "random"]:
        s = ws.split(kind)
        print(f"  {kind:6s} split sizes: " + ", ".join(f"{k}={len(v)}" for k, v in s.items()))
    if ws.use_cremad():
        c = ws.cremad()
        s = ws.split("cremad")
        print(f"CREMA-D: {len(c)} clips, {c.speaker.nunique()} speakers; "
              + ", ".join(f"{k}={len(v)} clips/{c.speaker[v].nunique()} spk" for k, v in s.items()))


def stage_cnn(ws):
    ws.ravdess()
    f = ws.cache / "ravdess_mfcc.pkl"
    if f.exists():
        feats = pickle.loads(f.read_bytes())
    else:
        print("Computing MFCCs...", flush=True)
        feats = [mfcc_features(w) for w in ws.rav_waves]
        f.write_bytes(pickle.dumps(feats))
    crem_feats = None
    for kind in ws.args.splits:
        for seed in ws.args.seeds:
            name = f"ravdess_{kind}_mfcc_cnn_seed{seed}"
            if ws.done(name):
                continue
            s, t0 = ws.split(kind), time.time()
            tr = CNNTrainer(8, ws.device, epochs=3 if ws.args.quick else 60)
            info = tr.fit([feats[i] for i in s["train"]], ws.rav_y[s["train"]],
                          [feats[i] for i in s["val"]], ws.rav_y[s["val"]], seed)
            test = metrics(ws.rav_y[s["test"]], tr.logits([feats[i] for i in s["test"]]).argmax(1), 8)
            cross = {}
            if kind == "actor":
                if ws.use_cremad() and crem_feats is None:
                    ws.cremad()
                    crem_feats = {i: mfcc_features(ws.crem_waves[i]) for i in ws.split("cremad")["test"]}
                cross = cross_eval_ravdess_model(ws, lambda idx: tr.logits([crem_feats[i] for i in idx]))
            ws.save(name, dict(dataset="ravdess", split=kind, model="mfcc_cnn", seed=seed, classes=D.RAVDESS_EMOTIONS,
                               sizes={k: len(v) for k, v in s.items()}, test=test, cross=cross,
                               train_seconds=time.time() - t0, **info))


def stage_frozen(ws):
    ws.ravdess()
    X = ws.pooled("ravdess")
    L, H = X.shape[1:]
    for kind in ws.args.splits:
        for seed in ws.args.seeds:
            name = f"ravdess_{kind}_wavlm_frozen_seed{seed}"
            if ws.done(name):
                continue
            s, t0 = ws.split(kind), time.time()
            tr = HeadTrainer(L, H, 8, ws.device, epochs=5 if ws.args.quick else 150)
            info = tr.fit(X[s["train"]], ws.rav_y[s["train"]], X[s["val"]], ws.rav_y[s["val"]], seed)
            test = metrics(ws.rav_y[s["test"]], tr.logits(X[s["test"]]).argmax(1), 8)
            cross = {}
            if kind == "actor":
                cross = cross_eval_ravdess_model(ws, lambda idx: tr.logits(ws.pooled("cremad")[idx]))
                if seed == ws.args.seeds[0]:
                    ck = ws.out / "checkpoints" / "wavlm_frozen_head.pt"
                    ck.parent.mkdir(parents=True, exist_ok=True)
                    torch.save(dict(mode="frozen", model_name=ws.args.model_name, classes=D.RAVDESS_EMOTIONS,
                                    state_dict=tr.model.state_dict(), test_ua=test["ua"], split="actor",
                                    seed=seed), ck)
            ws.save(name, dict(dataset="ravdess", split=kind, model="wavlm_frozen", seed=seed,
                               classes=D.RAVDESS_EMOTIONS, sizes={k: len(v) for k, v in s.items()}, test=test,
                               cross=cross, train_seconds=time.time() - t0, **info))


def stage_finetune(ws):
    ws.ravdess()
    W = ws.rav_waves
    for kind in ws.args.splits:
        for seed in ws.args.ft_seeds:
            name = f"ravdess_{kind}_wavlm_finetune{ws.args.tag}_seed{seed}"
            if ws.done(name):
                continue
            print(f"== {name}", flush=True)
            s, t0 = ws.split(kind), time.time()
            tr = FinetuneTrainer(8, ws.device, ws.args.model_name, epochs=1 if ws.args.quick else ws.args.ft_epochs,
                                 lr_encoder=ws.args.ft_lr, freeze_epochs=ws.args.ft_freeze_epochs)
            info = tr.fit([W[i] for i in s["train"]], ws.rav_y[s["train"]],
                          [W[i] for i in s["val"]], ws.rav_y[s["val"]], seed)
            test = metrics(ws.rav_y[s["test"]], tr.logits([W[i] for i in s["test"]]).argmax(1), 8)
            cross = {}
            if kind == "actor":
                cross = cross_eval_ravdess_model(ws, lambda idx: tr.logits([ws.crem_waves[i] for i in idx]))
                if ws.args.save_ft_checkpoint and seed == ws.args.ft_seeds[0]:
                    ck = ws.out / "checkpoints" / "wavlm_finetuned.pt"
                    ck.parent.mkdir(parents=True, exist_ok=True)
                    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in tr.model.state_dict().items()}
                    torch.save(dict(mode="finetune", model_name=ws.args.model_name, classes=D.RAVDESS_EMOTIONS,
                                    state_dict=sd, test_ua=test["ua"], split="actor", seed=seed), ck)
            ws.save(name, dict(dataset="ravdess", split=kind, model="wavlm_finetune" + ws.args.tag, seed=seed,
                               hparams=dict(epochs=ws.args.ft_epochs, lr_encoder=ws.args.ft_lr,
                                            freeze_epochs=ws.args.ft_freeze_epochs),
                               classes=D.RAVDESS_EMOTIONS, sizes={k: len(v) for k, v in s.items()}, test=test,
                               cross=cross, train_seconds=time.time() - t0, **info))
            del tr
            torch.cuda.empty_cache()


def stage_cremad(ws):
    """Train on CREMA-D (speaker-independent), test on unseen speakers and on RAVDESS."""
    ws.ravdess()
    ws.cremad()
    s = ws.split("cremad")
    y = ws.crem_y
    rav6 = ws.rav6_idx[:32] if ws.args.quick else ws.rav6_idx
    ry = ws.rav6_y[: len(rav6)]
    sizes = {k: len(v) for k, v in s.items()}

    X, RX = ws.pooled("cremad"), ws.pooled("ravdess")
    for seed in ws.args.seeds:
        name = f"cremad_speaker_wavlm_frozen_seed{seed}"
        if ws.done(name):
            continue
        t0 = time.time()
        tr = HeadTrainer(X.shape[1], X.shape[2], 6, ws.device, epochs=5 if ws.args.quick else 150)
        info = tr.fit(X[s["train"]], y[s["train"]], X[s["val"]], y[s["val"]], seed)
        test = metrics(y[s["test"]], tr.logits(X[s["test"]]).argmax(1), 6)
        cross = {"ravdess_all_actors": metrics(ry, tr.logits(RX[rav6]).argmax(1), 6)}
        ws.save(name, dict(dataset="cremad", split="speaker", model="wavlm_frozen", seed=seed,
                           classes=D.SHARED_EMOTIONS, sizes=sizes, test=test, cross=cross,
                           train_seconds=time.time() - t0, **info))

    Wc = ws.crem_waves
    for seed in ws.args.cremad_ft_seeds:
        name = f"cremad_speaker_wavlm_finetune{ws.args.tag}_seed{seed}"
        if ws.done(name):
            continue
        print(f"== {name}", flush=True)
        t0 = time.time()
        tr = FinetuneTrainer(6, ws.device, ws.args.model_name, epochs=1 if ws.args.quick else ws.args.cremad_ft_epochs,
                             lr_encoder=ws.args.ft_lr, freeze_epochs=ws.args.ft_freeze_epochs)
        info = tr.fit([Wc[i] for i in s["train"]], y[s["train"]], [Wc[i] for i in s["val"]], y[s["val"]], seed)
        test = metrics(y[s["test"]], tr.logits([Wc[i] for i in s["test"]]).argmax(1), 6)
        cross = {"ravdess_all_actors": metrics(ry, tr.logits([ws.rav_waves[i] for i in rav6]).argmax(1), 6)}
        ws.save(name, dict(dataset="cremad", split="speaker", model="wavlm_finetune" + ws.args.tag, seed=seed,
                           classes=D.SHARED_EMOTIONS, sizes=sizes, test=test, cross=cross,
                           train_seconds=time.time() - t0, **info))
        del tr
        torch.cuda.empty_cache()


STAGES = dict(prepare=stage_prepare, cnn=stage_cnn, frozen=stage_frozen, finetune=stage_finetune,
              cremad=stage_cremad)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--stages", nargs="+", default=list(STAGES), choices=list(STAGES))
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--out", default="results")
    ap.add_argument("--splits", nargs="+", default=["actor", "random"], choices=["actor", "random"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2], help="seeds for CNN and frozen runs")
    ap.add_argument("--ft-seeds", nargs="+", type=int, default=[0, 1, 2], help="seeds for RAVDESS fine-tuning")
    ap.add_argument("--cremad-ft-seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--model-name", default=SSL_MODEL)
    # Fine-tuning defaults were chosen on validation UA only (see docs/finetune_search.md).
    ap.add_argument("--ft-epochs", type=int, default=12)
    ap.add_argument("--ft-lr", type=float, default=3e-5, help="encoder learning rate")
    ap.add_argument("--ft-freeze-epochs", type=int, default=3, help="head-only epochs before unfreezing")
    ap.add_argument("--tag", default="", help="suffix for run names (keeps search runs separate)")
    ap.add_argument("--cremad-ft-epochs", type=int, default=8)
    ap.add_argument("--no-cremad", action="store_true", help="skip CREMA-D download and cross-corpus tests")
    ap.add_argument("--save-ft-checkpoint", action="store_true")
    ap.add_argument("--quick", action="store_true", help="tiny subsets and 1-5 epochs, for smoke tests only")
    args = ap.parse_args()
    ws = Workspace(args)
    print(f"device={ws.device} torch={torch.__version__}", flush=True)
    for st in args.stages:
        t0 = time.time()
        print(f"\n######## stage: {st}", flush=True)
        STAGES[st](ws)
        print(f"######## stage {st} finished in {(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/make_report.py
"""Aggregate results/runs/*.json into tables (Markdown + JSON) and figures."""
from __future__ import annotations

import argparse
import json
from collections import defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

MODELS = {"mfcc_cnn": "MFCC-CNN (baseline)", "wavlm_frozen": "WavLM Base+ frozen",
          "wavlm_finetune": "WavLM Base+ fine-tuned"}
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
ACTOR, RANDOM = "#2a78d6", "#eb6834"  # categorical slots 1-2 (validated adjacent pair)
BLUES = LinearSegmentedColormap.from_list("seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE})


def pct(m, s=None, n=1):
    return f"{100 * m:.1f}" if s is None or n < 2 else f"{100 * m:.1f} ± {100 * s:.1f}"


def aggregate(runs):
    groups = defaultdict(list)
    for r in runs:
        groups[(r["dataset"], r["split"], r["model"])].append(r)
    agg = {}
    for key, rs in groups.items():
        rs = sorted(rs, key=lambda r: r["seed"])
        a = dict(n=len(rs), seeds=[r["seed"] for r in rs], classes=rs[0]["classes"], sizes=rs[0]["sizes"])
        for m in ["ua", "wa", "macro_f1"]:
            v = np.array([r["test"][m] for r in rs])
            a[m], a[m + "_std"], a[m + "_runs"] = float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0, v.tolist()
        a["val_ua"] = float(np.mean([r["val_ua"] for r in rs]))
        a["confusion"] = np.sum([r["test"]["confusion"] for r in rs], axis=0).tolist()
        cross = defaultdict(list)
        for r in rs:
            for k, v in r.get("cross", {}).items():
                cross[k].append(v)
        a["cross"] = {k: dict(ua=float(np.mean([x["ua"] for x in v])), ua_std=float(np.std([x["ua"] for x in v], ddof=1))
                              if len(v) > 1 else 0.0, wa=float(np.mean([x["wa"] for x in v])), n=len(v),
                              confusion=np.sum([x["confusion"] for x in v], axis=0).tolist())
                      for k, v in cross.items()}
        if "layer_weights" in rs[0]:
            a["layer_weights"] = np.mean([r["layer_weights"] for r in rs], axis=0).tolist()
        agg["/".join(key)] = a
    return agg


def fig_split_comparison(agg, out):
    models = [m for m in MODELS if f"ravdess/actor/{m}" in agg and f"ravdess/random/{m}" in agg]
    if not models:
        return None
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    x, w = np.arange(len(models)), 0.36
    for off, split, color, label in [(-w / 2 - 0.01, "random", RANDOM, "Random split (speakers leak into test)"),
                                     (w / 2 + 0.01, "actor", ACTOR, "Actor split (unseen actors 21–24)")]:
        a = [agg[f"ravdess/{split}/{m}"] for m in models]
        vals = np.array([r["ua"] for r in a]) * 100
        errs = np.array([r["ua_std"] for r in a]) * 100
        ax.bar(x + off, vals, w, color=color, label=label, zorder=2)
        ax.errorbar(x + off, vals, yerr=errs, fmt="none", ecolor=INK2, elinewidth=1, capsize=3, zorder=3)
        for xi, v, e in zip(x + off, vals, errs):
            ax.text(xi, v + e + 1.2, f"{v:.1f}", ha="center", va="bottom", fontsize=9, color=INK)
    ax.axhline(12.5, color=INK2, lw=1, ls=(0, (3, 3)), zorder=1)
    ax.text(len(models) - 0.5, 13.5, "chance (8 classes)", ha="right", va="bottom", fontsize=8, color=INK2)
    ax.set_xticks(x, [MODELS[m] for m in models])
    ax.set_ylabel("Unweighted accuracy (%)")
    ax.set_ylim(0, 105)
    ax.yaxis.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(length=0)
    ax.legend(frameon=False, loc="upper left", fontsize=9)
    ax.set_title("RAVDESS, 8 emotions: random vs actor-independent split (mean ± sd over seeds)",
                 fontsize=10, color=INK, loc="left")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    return out


def fig_confusion(conf, classes, title, out):
    conf = np.array(conf, dtype=float)
    norm = conf / conf.sum(1, keepdims=True).clip(min=1) * 100
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(norm, cmap=BLUES, vmin=0, vmax=100)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{norm[i, j]:.0f}", ha="center", va="center", fontsize=8,
                    color="#ffffff" if norm[i, j] > 55 else INK)
    ax.set_xticks(range(len(classes)), classes, rotation=40, ha="right")
    ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, fontsize=10, loc="left")
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--include-quick", action="store_true")
    args = ap.parse_args()
    root = Path(args.results)
    runs = [json.loads(p.read_text()) for p in sorted((root / "runs").glob("*.json"))]
    runs = [r for r in runs if args.include_quick or not r.get("quick")]
    if not runs:
        raise SystemExit("no runs found")
    agg = aggregate(runs)
    (root / "summary.json").write_text(json.dumps(agg, indent=1))
    figs = root / "figures"
    figs.mkdir(exist_ok=True)

    L = ["# Results", "",
         "All numbers are on held-out test data, scored once with the checkpoint picked on validation UA.",
         "UA = unweighted accuracy (mean per-class recall); WA = plain accuracy. Mean ± sd over seeds.", "",
         "## RAVDESS (8 emotions): random split vs actor-independent split", "",
         "| Model | Random split UA | Actor split UA (unseen actors) | Gap (pp) | Actor split WA | Actor split macro-F1 | Seeds |",
         "|---|---|---|---|---|---|---|"]
    for m, label in MODELS.items():
        a, r = agg.get(f"ravdess/actor/{m}"), agg.get(f"ravdess/random/{m}")
        if not a:
            continue
        gap = f"{100 * (r['ua'] - a['ua']):+.1f}" if r else "–"
        L.append(f"| {label} | {pct(r['ua'], r['ua_std'], r['n']) if r else '–'} | **{pct(a['ua'], a['ua_std'], a['n'])}** "
                 f"| {gap} | {pct(a['wa'], a['wa_std'], a['n'])} | {pct(a['macro_f1'], a['macro_f1_std'], a['n'])} | {a['n']} |")
    L += ["", "Split sizes (both splits): train 1080 / val 120 / test 240 clips. "
          "Actor split: actors 1–18 train, 19–20 val, 21–24 test.", ""]

    if fig_split_comparison(agg, figs / "split_comparison.png"):
        L += ["![split comparison](figures/split_comparison.png)", ""]

    L += ["## Per-class recall on unseen actors (actor split, pooled over seeds)", ""]
    first = next((agg[k] for k in (f"ravdess/actor/{m}" for m in MODELS) if k in agg), None)
    if first:
        classes = first["classes"]
        L += ["| Model | " + " | ".join(classes) + " |", "|---|" + "---|" * len(classes)]
        for m, label in MODELS.items():
            a = agg.get(f"ravdess/actor/{m}")
            if a:
                c = np.array(a["confusion"], float)
                rec = np.diag(c) / c.sum(1).clip(min=1)
                L.append(f"| {label} | " + " | ".join(f"{100 * v:.0f}" for v in rec) + " |")
                fig_confusion(a["confusion"], classes, f"{label}: unseen actors (row %)",
                              figs / f"confusion_actor_{m}.png")
        L += [""] + [f"![{m}](figures/confusion_actor_{m}.png)" for m in MODELS if f"ravdess/actor/{m}" in agg] + [""]

    cross_rows = []
    for m, label in MODELS.items():
        a = agg.get(f"ravdess/actor/{m}")
        if a and "cremad_test" in a["cross"]:
            c = a["cross"]["cremad_test"]
            cross_rows.append(f"| {label} | RAVDESS actors 1–18 | RAVDESS actors 21–24 (8-class) | {pct(a['ua'], a['ua_std'], a['n'])} "
                              f"| CREMA-D, 18 unseen speakers (6-class) | {pct(c['ua'], c['ua_std'], c['n'])} |")
    for m, label in MODELS.items():
        a = agg.get(f"cremad/speaker/{m}")
        if a:
            c = a["cross"]["ravdess_all_actors"]
            cross_rows.append(f"| {label} | CREMA-D, 64 speakers | CREMA-D, 18 unseen speakers (6-class) | {pct(a['ua'], a['ua_std'], a['n'])} "
                              f"| RAVDESS, all 24 actors (6-class) | {pct(c['ua'], c['ua_std'], c['n'])} |")
            fig_confusion(c["confusion"], a["classes"], f"{label}: trained CREMA-D → tested RAVDESS",
                          figs / f"confusion_cremad_to_ravdess_{m}.png")
    if cross_rows:
        L += ["## Cross-corpus generalisation (RAVDESS ↔ CREMA-D)", "",
              "Only the 6 emotions both corpora share are scored across corpora (neutral, happy, sad, angry, fearful, disgust); "
              "an 8-class RAVDESS model is restricted to those 6 outputs. Chance = 16.7%.", "",
              "| Model | Trained on | In-corpus test | UA | Cross-corpus test | UA |", "|---|---|---|---|---|---|"] + cross_rows + [""]

    lw = agg.get("ravdess/actor/wavlm_frozen", {}).get("layer_weights")
    if lw:
        top = np.argsort(lw)[::-1][:3]
        L += ["## Which WavLM layers matter (frozen model, learned layer weights)", "",
              "Layer 0 = CNN/embedding output, 12 = last transformer layer. Top layers: "
              + ", ".join(f"{i} ({100 * lw[i]:.0f}%)" for i in top) + ".", ""]
    (root / "RESULTS.md").write_text("\n".join(L), encoding="utf-8")
    print("\n".join(L))


if __name__ == "__main__":
    main()

## 1. Data: RAVDESS (Zenodo) and CREMA-D (official GitHub LFS)

In [ ]:
run('--stages prepare')

## 2. Baseline: MFCC + small CNN (3 seeds × actor/random split)

In [ ]:
run('--stages cnn')

## 3. WavLM Base+ frozen, SUPERB-style layer-weighted head (3 seeds × 2 splits)

In [ ]:
run('--stages frozen')

## 4. WavLM Base+ fine-tuned (3 seeds × 2 splits, 20 epochs each)

In [ ]:
run('--stages finetune --save-ft-checkpoint')

## 5. Cross-corpus: train on CREMA-D speakers, test on unseen speakers and on RAVDESS

In [ ]:
run('--stages cremad')

## 6. Report

In [ ]:
!python scripts/make_report.py

In [ ]:
# Bundle the run files and figures (small) to copy back into the repo.
!tar czf results_bundle.tgz results/runs results/figures results/RESULTS.md results/summary.json
from google.colab import files
files.download('results_bundle.tgz')
# Optional, ~190 MB: the fine-tuned checkpoint for the Hugging Face Space demo.
# files.download('results/checkpoints/wavlm_finetuned.pt')